# One-vs-one ensemble (15 binary CNNs + soft voting)

A binary CNN for **every pair** of emotions — all `C(6,2) = 15` combinations (Angry–Disgusted, Angry–Fearful, … Neutral–Sad). Every emotion is directly compared against each of the other five, so each emotion appears in exactly **5** models.

Each model outputs a sigmoid P(a). At test time every clip is scored by all 15 models; each model adds P(a) to emotion `a` and 1−P(a) to emotion `b`. Summed across the 15 models (each emotion accumulating from its 5 pairs), the **highest total wins** — the soft-voting version of "most votes wins".

This is the full one-vs-one scheme — strictly more discriminative than the 6-pair cycle, since non-adjacent emotions (e.g. Angry vs Happy) now get their own dedicated model, at the cost of 15 models to train instead of 6.

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

EMO_DIR = Path("../Emotions")

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df = pd.DataFrame(rows)

# Keep only CREMA-D + RAVDESS, drop the Surprised emotion (folder spelled 'Suprised')
df = df[df["source"].isin(["CREMA-D", "RAVDESS"])]
df = df[df["emotion"] != "Suprised"].reset_index(drop=True)
print(f"{len(df)} clips, {df['emotion'].nunique()} emotions")
df["emotion"].value_counts()

In [ ]:
import librosa
from tqdm.auto import tqdm

# Raw clip duration; keep only 1–5s clips
df["duration"] = [librosa.get_duration(path=p) for p in tqdm(df["path"], desc="duration")]
before = len(df)
df = df[(df["duration"] >= 1.0) & (df["duration"] <= 5.0)].reset_index(drop=True)
print(f"removed {before - len(df)} clips  ->  {len(df)} remain")
df["emotion"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

# 85/15 split, stratified by emotion
dftrain, dftest = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df["emotion"])
dftrain = dftrain.reset_index(drop=True)
dftest  = dftest.reset_index(drop=True)
print(f"train: {len(dftrain)}   test: {len(dftest)}")

In [ ]:
SR         = 22050
N_MFCC     = 40
HOP        = 512
MAX_FRAMES = int(np.ceil(5 * SR / HOP))   # 216 frames -> covers the 5s cap

def _fit2d(m):
    if m.shape[1] < MAX_FRAMES:
        return np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])))
    return m[:, :MAX_FRAMES]

def _fit1d(a):
    a = a[:MAX_FRAMES] if len(a) >= MAX_FRAMES else np.pad(a, (0, MAX_FRAMES - len(a)))
    return np.tile(a, (N_MFCC, 1))

def features(path):
    y, sr = librosa.load(path, sr=SR)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP)
    d1   = librosa.feature.delta(mfcc)
    d2   = librosa.feature.delta(mfcc, order=2)
    rms  = librosa.feature.rms(y=y, hop_length=HOP)[0]
    f0   = librosa.yin(y, fmin=65, fmax=2093, sr=sr, hop_length=HOP)
    zcr  = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]
    cen  = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP)[0]
    return np.stack([_fit2d(mfcc), _fit2d(d1), _fit2d(d2),
                     _fit1d(rms), _fit1d(f0), _fit1d(zcr), _fit1d(cen)], axis=-1)

def build_X(frame, desc):
    return np.stack([features(p) for p in tqdm(frame["path"], desc=desc)]).astype("float32")

# Extract per split (no leakage)
X_train = build_X(dftrain, "train")
X_test  = build_X(dftest,  "test")
print("X_train:", X_train.shape, " X_test:", X_test.shape)   # (n, 40, 216, 7)

In [ ]:
# Per-channel standardization using TRAIN stats only
axes = (0, 1, 2)
mean = X_train.mean(axis=axes, keepdims=True)
std  = X_train.std(axis=axes, keepdims=True)
X_train = (X_train - mean) / (std + 1e-9)
X_test  = (X_test  - mean) / (std + 1e-9)
print("standardized — X_train:", X_train.shape, " X_test:", X_test.shape)

In [ ]:
from itertools import combinations
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

EMOTIONS = ["Angry", "Disgusted", "Fearful", "Happy", "Neutral", "Sad"]
PAIRS = list(combinations(EMOTIONS, 2))           # all 15 one-vs-one pairs
print(f"{len(PAIRS)} pairwise models:", PAIRS)

def make_binary_model(input_shape):
    m = keras.Sequential([
        keras.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),          # P(first emotion of the pair)
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

models, pair_scores = {}, {}
for i, (a, b) in enumerate(PAIRS, 1):
    tr_mask = dftrain["emotion"].isin([a, b]).values        # train on the two emotions only
    Xp = X_train[tr_mask]
    yp = (dftrain["emotion"][tr_mask] == a).astype("float32").values   # 1 = a, 0 = b

    # explicit shuffled + STRATIFIED 85/15 val split so both classes appear in validation
    Xtr, Xval, ytr, yval = train_test_split(
        Xp, yp, test_size=0.15, random_state=42, shuffle=True, stratify=yp)

    print(f"\n===== [{i}/{len(PAIRS)}] {a} vs {b}  (train: {len(ytr)}, val: {len(yval)}, {a}: {int(yp.sum())}) =====")
    m = make_binary_model(X_train.shape[1:])
    m.fit(Xtr, ytr, epochs=30, batch_size=32, verbose=0,
          validation_data=(Xval, yval),
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                                   restore_best_weights=True)])
    models[(a, b)] = m
    pair_scores[(a, b)] = m.predict(X_test, verbose=0).ravel()   # P(a) for EVERY test clip
    print(f"  done — mean P({a}) on test: {pair_scores[(a, b)].mean():.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns, matplotlib.pyplot as plt

# Per-model inspection: each binary model on its own two emotions' test clips
THRESH   = 0.5                                  # P(a) >= THRESH -> predict a, else b
true_emo = dftest["emotion"].values

ncol = 5
nrow = int(np.ceil(len(PAIRS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 4 * nrow))
axes = axes.ravel()

for ax, (a, b) in zip(axes, PAIRS):
    te_mask = np.isin(true_emo, [a, b])         # only this pair's test clips
    labels  = [b, a]                            # 0 = b, 1 = a

    y_true = (true_emo[te_mask] == a).astype(int)
    y_pred = (pair_scores[(a, b)][te_mask] >= THRESH).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=labels, yticklabels=labels, ax=ax)
    acc = (y_true == y_pred).mean()
    ax.set_title(f"{a} vs {b}  (acc {acc:.2f})")
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("True", fontweight="bold")
    ax.tick_params(axis="y", rotation=0)

    print(f"===== {a} vs {b}  ({te_mask.sum()} test clips) =====")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

for ax in axes[len(PAIRS):]:                     # hide any unused panels
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

idx = {e: i for i, e in enumerate(EMOTIONS)}

# Soft voting: each of the 15 models adds P(a) to emotion a and 1-P(a) to emotion b
# (each emotion accumulates from the 5 pairs it appears in)
vote = np.zeros((len(dftest), len(EMOTIONS)))
for (a, b), pa in pair_scores.items():
    vote[:, idx[a]] += pa
    vote[:, idx[b]] += 1 - pa

pred_emo = np.array(EMOTIONS)[vote.argmax(1)]
true_emo = dftest["emotion"].values

print(f"evaluating on all {len(true_emo)} test clips across {len(EMOTIONS)} emotions "
      f"({len(PAIRS)} one-vs-one models)\n")
print(classification_report(true_emo, pred_emo, labels=EMOTIONS, zero_division=0))

cm = confusion_matrix(true_emo, pred_emo, labels=EMOTIONS)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=EMOTIONS, yticklabels=EMOTIONS, cmap="Blues")
plt.xlabel("Predicted", fontweight="bold")
plt.ylabel("True", fontweight="bold")
plt.title("One-vs-one ensemble (15 models) — soft voting (highest total wins)")
plt.tight_layout()
plt.show()